# Deploying and Using MedImageParse3D Model for Inference using Batch Endpoints
This example illustrates how to deploy MedImageParse3D, a state-of-the-art 3D volumetric segmentation model tailored for biomedical imaging. For this Notebook, we use Python 3.10, AzureML v2.

### Task
The primary task is 3D volumetric semantic segmentation, where the goal is to identify and label specific regions within a 3D medical volume based on their semantic meaning using a submitted volume and a text prompt.
 
### Model
MedImageParse3D is powered by a BoltzFormer transformer-based architecture, fine-tuned for 3D segmentation tasks on extensive biomedical volumetric datasets. It is designed to excel in handling complex segmentation challenges across diverse 3D imaging modalities. 

### Inference data
For this demonstration, we will use 3D NIfTI volumes (e.g., CT or MRI scans) and perform volumetric segmentation of anatomical structures and pathologies.

### Outline
1. Setup pre-requisites
2. Pick a model to deploy
3. Deploy the model to a batch endpoint
4. Test the endpoint
5. Clean up resources - delete the endpoint


## 1. Setup pre-requisites
* Install [Azure ML Client library for Python](https://learn.microsoft.com/en-us/python/api/overview/azure/ai-ml-readme?view=azure-python)
* Connect to AzureML Workspace and authenticate.

In [ ]:
from azure.ai.ml import MLClient, Input
from azure.ai.ml.entities import (
    BatchEndpoint,
    ModelBatchDeployment,
    ModelBatchDeploymentSettings,
    Model,
    AmlCompute,
    Data,
    BatchRetrySettings,
    CodeConfiguration,
    Environment,
)
from azure.ai.ml.constants import AssetTypes, BatchDeploymentOutputAction
from azure.identity import DefaultAzureCredential
import pandas as pd

credential = DefaultAzureCredential()
ml_workspace = MLClient.from_config(credential)
print("Workspace:", ml_workspace)
ml_registry = MLClient(credential, registry_name="azureml")
print("Registry:", ml_registry)

## 2. Pick a model to deploy

In this example, we use the `MedImageParse3D` model. If you have opened this notebook for a different model, replace the model name accordingly.


In [ ]:
model = ml_registry.models.get(name="MedImageParse3D", label="latest")
model


## 3. Deploy the model to a batch endpoint
Batch endpoints process large volumes of data asynchronously, ideal for scoring many 3D NIfTI volumes.

The steps below show how to deploy a batch endpoint programmatically. You can skip the steps in this section if you just want to test an existing endpoint.


### Create compute cluster

In [ ]:
compute_name = "mip3d-batch-cluster"
if not any(filter(lambda m: m.name == compute_name, ml_workspace.compute.list())):
    compute_cluster = AmlCompute(
        name=compute_name,
        description="GPU cluster compute for MedImageParse3D batch inference",
        min_instances=0,
        max_instances=1,
        size="Standard_NC40ads_H100_v5",
    )
    ml_workspace.compute.begin_create_or_update(compute_cluster).result()


### Create batch endpoint

In [ ]:
import random
import string

endpoint_prefix = "mip3d-batch"
endpoint_list = list(
    filter(
        lambda m: m.name.startswith(endpoint_prefix),
        ml_workspace.batch_endpoints.list(),
    )
)

if endpoint_list:
    endpoint = endpoint_list and endpoint_list[0]
    print("Found existing endpoint:", endpoint.name)
else:
    # Creating a unique endpoint name by including a random suffix
    allowed_chars = string.ascii_lowercase + string.digits
    endpoint_suffix = "".join(random.choice(allowed_chars) for x in range(5))
    endpoint_name = f"{endpoint_prefix}-{endpoint_suffix}"
    endpoint = BatchEndpoint(
        name=endpoint_name,
        description="A batch endpoint for scoring 3D volumes from MedImageParse3D.",
        tags={"type": "medimageparse3d"},
    )
    ml_workspace.begin_create_or_update(endpoint).result()
    print(f"Created new endpoint: {endpoint_name}")


### Deploy MedImageParse3D to batch endpoint

- **max_concurrency_per_instance**: Determines the number of worker processes to spawn. Each worker process loads the model into GPU. 3D volumes require significantly more GPU memory than 2D images, so concurrency may need to be reduced.
- **retry_settings**: Timeout may need to be adjusted based on volume size. Larger 3D volumes require longer timeout; otherwise, worker process may end prematurely.


In [ ]:
deployment = ModelBatchDeployment(
    name="mip3d-dpl",
    description="A deployment for model MedImageParse3D",
    endpoint_name=endpoint.name,
    model=model,
    compute=compute_name,
    settings=ModelBatchDeploymentSettings(
        max_concurrency_per_instance=1,  # 3D volumes need more GPU memory per instance
        mini_batch_size=1,
        instance_count=1,
        output_action=BatchDeploymentOutputAction.APPEND_ROW,
        output_file_name="predictions.csv",
        retry_settings=BatchRetrySettings(max_retries=3, timeout=600),  # Higher timeout for 3D
        logging_level="info",
    ),
)
ml_workspace.begin_create_or_update(deployment).result()


In [ ]:
# Check deployment logs (optional debugging step)
# ml_workspace.batch_deployments.get_logs(name=deployment.name, endpoint_name=endpoint.name, lines=50)


In [ ]:
endpoint = ml_workspace.batch_endpoints.get(endpoint.name)
endpoint.defaults.deployment_name = deployment.name
ml_workspace.batch_endpoints.begin_create_or_update(endpoint).result()
print(f"The default deployment is {endpoint.defaults.deployment_name}")

## 4 Test the endpoint - base64 encoded image and text

### Load test dataset

Please follow the data download instructions in the main [README](../../README.md) to download the sample data for this notebook.

In [ ]:
import glob
import os

data_root = "/home/azureuser/data/healthcare-ai"  # Change to the location you downloaded the data
root_dir = os.path.join(data_root, "medimageparse3d-volumes")

nifti_files = glob.glob(f"{root_dir}/**/*.nii.gz", recursive=True)
print(f"Found {len(nifti_files)} NIfTI files")


### Create the input CSV file

#### Zero-Padding Batch Filenames

>In the example below only one batch is created (batch_input_001.csv). If you need to create more batches please make sure to use the zero-padding batch index.

For more detailed example please refer to the MedImageInsight notebook `mi2-deploy-batch-endpoint.ipynb`.   
In that notebook, the function `write_to_csv()` will automatically create batch files with  **zero-padded numeric suffixes** (e.g., `batch_input_001.csv`, `batch_input_002.csv`, ..., `batch_input_010.csv`). 
It's essential to use that index for enumerating your batches. 

This ensures that files are **sorted in the correct numerical order**, rather than lexicographic string order. E.g., without padding, `batch10` would appear **before** `batch2` or `batch3` when sorting, which can lead to confusing or incorrect alignment between batch input files and batch output results. Zero-padding helps maintain predictable ordering and avoids mismatches during downstream processing or aggregation.

In [ ]:
import base64
import os


def read_base64_volume(volume_path):
    with open(volume_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


data = []
for f in nifti_files:
    base64_volume = read_base64_volume(f)
    data.append([base64_volume, "tumor"])

csv_path = os.path.join(os.getcwd(), "batch_input_001.csv")
df_input = pd.DataFrame(data, columns=["image", "text"])
df_input.to_csv(csv_path)


### Load the test dataset into AzureML

In [ ]:
dataset_name = "mip3d-nifti-dataset"

nifti_dataset = Data(
    path=csv_path,
    type=AssetTypes.URI_FILE,
    description="A dataset of 3D NIfTI volumes for MedImageParse3D batch inference",
    name=dataset_name,
)

ml_workspace.data.create_or_update(nifti_dataset)


### Verify the test dataset is uploaded successfully

In [ ]:
ml_workspace.data.get(name=dataset_name, label="latest")

### Submit a job to the batch endpoint

In [ ]:
input = Input(type=AssetTypes.URI_FILE, path=nifti_dataset.path)
input


In [ ]:
job = ml_workspace.batch_endpoints.invoke(endpoint_name=endpoint.name, input=input)

In [ ]:
# Monitor job progress
ml_workspace.jobs.stream(job.name)

### Download the job output
MedImageParse3D segmentation results can be found in file `named-outputs/score/predictions.csv`


In [ ]:
scoring_job = list(ml_workspace.jobs.list(parent_job_name=job.name))[0]
scoring_job

In [ ]:
ml_workspace.jobs.download(
    name=scoring_job.name, download_path=".", output_name="score"
)

### Load job result

In [ ]:
pred_csv_path = os.path.join("named-outputs", "score", "predictions.csv")
df_result = pd.read_csv(pred_csv_path, header=None)
print("df_result.shape:", df_result.shape)
print(df_result.iloc[0])  # print first row

### Display job result

In [ ]:
import base64
import json
import matplotlib.pyplot as plt
import numpy as np
import nibabel as nib


def parse_image(json_encoded):
    """Decode an image pixel data array in JSON.
    Return image pixel data as an array.
    """
    array_metadata = json.loads(json_encoded)
    base64_encoded = array_metadata["data"]
    shape = tuple(array_metadata["shape"])
    dtype = np.dtype(array_metadata["dtype"])
    array_bytes = base64.b64decode(base64_encoded)
    array = np.frombuffer(array_bytes, dtype=dtype).reshape(shape)
    return array


def parse_labels(s):
    return json.loads(s.replace("'", '"'))


def plot_3d_segmentation_slices(volume, segmentation_masks, labels=None, slice_idx=None, axis=2):
    """Plot a central slice from the 3D volume with segmentation mask overlays.

    Args:
        volume: 3D numpy array from a NIfTI file.
        segmentation_masks: list of 3D numpy arrays (one per label).
        labels: list of label strings for each mask.
        slice_idx: index of the slice to display (default: middle slice).
        axis: axis along which to take the slice (0=sagittal, 1=coronal, 2=axial).
    """
    if slice_idx is None:
        slice_idx = volume.shape[axis] // 2

    vol_slice = np.take(volume, slice_idx, axis=axis)
    vol_slice = ((vol_slice - vol_slice.min()) / (vol_slice.max() - vol_slice.min() + 1e-8) * 255).astype(np.uint8)

    n_masks = len(segmentation_masks)
    fig, ax = plt.subplots(1, n_masks + 1, figsize=(5 * (n_masks + 1), 5))
    if n_masks == 0:
        ax = [ax]
    ax[0].imshow(vol_slice, cmap="gray")
    ax[0].set_title(f"Original (slice {slice_idx})")
    ax[0].axis("off")

    for i, mask in enumerate(segmentation_masks):
        mask_slice = np.take(mask, min(slice_idx, mask.shape[axis] - 1), axis=axis)
        ax[i + 1].imshow(vol_slice, cmap="gray")
        title = labels[i] if labels and i < len(labels) else f"Mask {i+1}"
        ax[i + 1].set_title(title)
        overlay = np.zeros((*mask_slice.shape, 4), dtype=np.float32)
        overlay[mask_slice > 0] = [1.0, 0.0, 0.0, 0.6]
        ax[i + 1].imshow(overlay)
        ax[i + 1].axis("off")
    plt.tight_layout()
    plt.show()


In [ ]:
for index in range(len(df_input)):
    nii_path = nifti_files[index]
    vol = nib.load(nii_path).get_fdata()
    result = df_result.iloc[index]

    segmentation_masks_encoded = parse_image(result.iloc[1])
    labels = parse_labels(result.iloc[2])

    # Separate individual masks if stacked (4D → list of 3D)
    if segmentation_masks_encoded.ndim == 4:
        masks = [segmentation_masks_encoded[i] for i in range(segmentation_masks_encoded.shape[0])]
    else:
        masks = [segmentation_masks_encoded]

    print(f"Volume {index}: {os.path.basename(nii_path)}, labels: {labels}")
    plot_3d_segmentation_slices(vol, masks, labels)


## 5. Clean up resources - delete the batch endpoint
Don't forget to delete the batch endpoint when done, else compute may remain allocated.


In [ ]:
ml_workspace.batch_endpoints.begin_delete(endpoint.name).result()
